# 02. 탐색적 데이터 분석 (EDA)

- 입력: `data/processed/seoul_bike_clean.csv` (01에서 저장)
- 하는 일: 타깃 분포, 시간대·계절·기온·비/눈·공휴일/주말별 대여 수, 변수 간 상관관계, humidity 0 이상치 확인
- 출력: 없음 (그래프와 표만 확인, 데이터는 변경하지 않음)

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트(notebooks/의 상위 폴더)를 import 경로에 추가해서 src 패키지를 사용
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import pandas as pd

from src.data import load_clean

df = load_clean()
print(df.shape)

In [ ]:
import matplotlib.pyplot as plt

# 운영하지 않은 시간(functioning_day == 'No')은 대여 수가 전부 0이라 평균을 왜곡하므로
# 수요 관련 그래프는 운영한 시간만으로 그림 (df 자체는 변경하지 않음)
op = df[df['functioning_day'] == 'Yes']
print(df['functioning_day'].value_counts())
print('운영 시간 중 대여 수 0인 행:', (op['rented_bike_count'] == 0).sum())
op.describe()

In [ ]:
# 타깃 분포
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(op['rented_bike_count'], bins=50)
ax.set_xlabel('Rented bike count (per hour)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of hourly rentals')
plt.show()
print('skewness:', round(op['rented_bike_count'].skew(), 3))

In [ ]:
# 시간대별 평균 대여 수
hourly = op.groupby('hour')['rented_bike_count'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly.index, hourly.values, marker='o')
ax.set_xticks(range(24))
ax.set_xlabel('Hour')
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by hour')
ax.grid(alpha=0.3)
plt.show()
hourly.round(1)

In [ ]:
# 계절별 평균 대여 수
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
seasonal = op.groupby('seasons')['rented_bike_count'].mean().reindex(season_order)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(seasonal.index, seasonal.values)
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by season')
plt.show()
seasonal.round(1)

In [ ]:
# 기온과 대여 수
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(op['temperature'], op['rented_bike_count'], s=3, alpha=0.3)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Rented bike count')
ax.set_title('Temperature vs rentals')
plt.show()

# 5°C 구간별 평균
temp_bin = pd.cut(op['temperature'], bins=range(-20, 45, 5))
op.groupby(temp_bin, observed=True)['rented_bike_count'].agg(['mean', 'count']).round(1)

In [ ]:
# 비·눈 여부에 따른 평균 대여 수
weather = pd.DataFrame({
    'rain': op.groupby(op['rainfall'] > 0)['rented_bike_count'].mean(),
    'snow': op.groupby(op['snowfall'] > 0)['rented_bike_count'].mean(),
}).rename(index={False: 'No', True: 'Yes'})

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, col in zip(axes, ['rain', 'snow']):
    ax.bar(weather.index, weather[col])
    ax.set_title(f'{col} > 0')
axes[0].set_ylabel('Mean rented bike count')
plt.show()
print('rainfall > 0 시간:', (op['rainfall'] > 0).sum(), '/ snowfall > 0 시간:', (op['snowfall'] > 0).sum())
weather.round(1)

In [ ]:
# 공휴일/평일, 주말/주중의 시간대별 패턴 (요일은 그래프용으로만 계산, df에는 추가하지 않음)
day_type = (op['date'].dt.dayofweek >= 5).map({False: 'Weekday', True: 'Weekend'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for label, g in op.groupby('holiday'):
    axes[0].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
for label, g in op.groupby(day_type):
    axes[1].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
axes[0].set_title('Holiday vs No Holiday')
axes[1].set_title('Weekday vs Weekend')
for ax in axes:
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel('Hour')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Mean rented bike count')
plt.show()

print(op.groupby('holiday')['rented_bike_count'].mean().round(1))
print(op.groupby(day_type)['rented_bike_count'].mean().round(1))

In [ ]:
# 수치형 변수 간 상관관계
corr = op.select_dtypes('number').corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()
corr['rented_bike_count'].drop('rented_bike_count').sort_values(key=abs, ascending=False).round(3)

In [ ]:
# humidity 최솟값 0 확인 (이상치 여부)
print('humidity == 0 행 수:', (df['humidity'] == 0).sum())
print('humidity 하위 값:', sorted(df['humidity'].unique())[:10])
df[df['humidity'] == 0][['date', 'hour', 'temperature', 'humidity', 'dew_point_temp', 'rainfall', 'rented_bike_count']]